In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import json

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 28
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## CIVICUS Monitor Pipeline

**Source:** CIVICUS Monitor
**Access:** Automated REST API — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — CIVICUS section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Civic space rating (Open/Narrowed/Obstructed/Repressed/Closed) | Political participation | Primary tier 1 |
| Civic space rating | Civil society space | Primary tier 1 |

In [3]:
import requests
import json
import pandas as pd
from datetime import datetime

CIVICUS_API_URL = "https://monitor.civicus.org/api/countries/"

print("Downloading CIVICUS Monitor ratings from API...")
response = requests.get(CIVICUS_API_URL, headers=BROWSER_HEADERS, timeout=30)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")

data = response.json()
print(f"Countries: {len(data)}")

# Flatten to country-year — take latest rating per calendar year
# Multiple interim updates per year exist; keep the latest entry for each year
records = []
for country in data:
    country_name = country['name']
    for rating_entry in country['ratings']:
        # Parse timestamp to extract year
        created_at = pd.to_datetime(rating_entry['created_at'], utc=True)
        records.append({
            'country_name': country_name,
            'year':         created_at.year,
            'created_at':   created_at,
            'civicus_score':  float(rating_entry['score']),
            'civicus_rating': rating_entry['rating'],
        })

# Build DataFrame
df = pd.DataFrame(records)

# Keep only the latest entry per country-year
df = df.sort_values('created_at').groupby(['country_name', 'year']).last().reset_index()
df = df.drop(columns=['created_at'])

# Filter to framework start year
civicus = df[df['year'] >= FRAMEWORK_START_YEAR].copy()
civicus = civicus.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"\nShape: {civicus.shape}")
print(f"Years: {civicus['year'].min()} — {civicus['year'].max()}")
print(f"Countries: {civicus['country_name'].nunique()}")
print(f"\nRating distribution:")
print(civicus['civicus_rating'].value_counts())
print(civicus.head(3))

Status: 200, Size: 251.9KB
Countries: 199

Shape: (674, 4)
Years: 2022 — 2025
Countries: 198

Rating distribution:
civicus_rating
Repressed     180
Narrowed      135
Obstructed    135
Open          116
Closed        108
Name: count, dtype: int64
  country_name  year  civicus_score civicus_rating
0  Afghanistan  2022          30.15      Repressed
1  Afghanistan  2023          12.16         Closed
2  Afghanistan  2024          11.00         Closed


In [4]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "civicus_clean.csv")
civicus.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {civicus.shape}")

# Derive data currency — no hardcoding
latest_year = str(int(civicus['year'].max()))

# Update download log
update_entry(
    "CIVICUS",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="civicus_clean.csv",
    latest_available_version=latest_year,
    notes="Automated via CIVICUS Monitor REST API (monitor.civicus.org/api/countries/). "
          "Latest rating per calendar year retained from multiple interim updates. "
          "5-point categorical rating: Open/Narrowed/Obstructed/Repressed/Closed. "
          "Note: API only returns data from 2022 onwards despite CIVICUS launching 2016. "
          "Coverage: 199 countries, 2022-present."
)
print_entry("CIVICUS")

Written: /Users/boulanger/Documents/governance-framework/data/processed/civicus_clean.csv
Shape: (674, 4)
[download_log] Updated entry for CIVICUS
  source_id: CIVICUS
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2025
  local_filename: civicus_clean.csv
  latest_available_version: 2025
  no_update_reason: nan
  notes: Automated via CIVICUS Monitor REST API (monitor.civicus.org/api/countries/). Latest rating per calendar year retained from multiple interim updates. 5-point categorical rating: Open/Narrowed/Obstructed/Repressed/Closed. Note: API only returns data from 2022 onwards despite CIVICUS launching 2016. Coverage: 199 countries, 2022-present.
